# Week 10 – Baseline Backtest

This notebook backtests two baseline strategies on the RAATS daily bundle:
1. Buy‑and‑hold
2. SMA crossover (50‑day vs 200‑day)

Performance metrics are computed using the functions in `src/evaluation/metrics.py`.


In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('.')  # allow import from src
from src.evaluation.metrics import compute_sharpe, compute_max_drawdown, compute_win_rate
from zipline.api import order_target, symbol, set_slippage, set_commission, slippage, commission
from zipline import run_algorithm
from zipline.pipeline import USEquityPricing
from zipline.pipeline.factors import SimpleMovingAverage
from zipline.filters = static


In [ ]:
# Helper: compute buy‑and‑hold equity curve for a given symbol
def buy_and_hold(context, data):
    # On first day, go fully invested; thereafter hold
    if not hasattr(context, 'bought'):
        order_target(context.asset, context.portfolio.portfolio_value / data.current(context.asset, 'price'))
        context.bought = True


In [ ]:
# Helper: SMA crossover strategy
def sma_crossover_initialize(context):
    context.sma_50 = SimpleMovingAverage(inputs=[USEquityPricing.close], window_length=50, mask=static())
    context.sma_200 = SimpleMovingAverage(inputs=[USEquityPricing.close], window_length=200, mask=static())
    context.asset = symbol('AAPL')
    set_slippage(slippage.FixedSlippage(spread=0))
    set_commission(commission.PerShare(cost=0.005))


In [ ]:
def sma_crossover_handle_data(context, data):
    if not data.can_trade(context.asset):
        return
    sma_50 = context.sma_50[data.date][context.asset]
    sma_200 = context.sma_200[data.date][context.asset]
    if np.isnan(sma_50) or np.isnan(sma_200):
        return
    if sma_50 > sma_200:
        target = 1.0  # fully invested
    else:
        target = 0.0  # out of market
    order_target(context.asset, target * context.portfolio.portfolio_value / data.current(context.asset, 'price'))


In [ ]:
# Run backtests
start = pd.Timestamp('2015-01-01', tz='utc')
end   = pd.Timestamp('2024-12-31', tz='utc')

# Buy‑and‑hold
result_bh = run_algorithm(
    start=start,
    end=end,
    initialize=lambda c: setattr(c, 'asset', symbol('AAPL')) or setattr(c, 'bought', False),
    handle_data=buy_and_hold,
    capital_base=10_000,
    bundle='raats_daily'
)
equity_bh = result_bh.portfolio_value
returns_bh = result_bh.returns

# SMA crossover
result_sma = run_algorithm(
    start=start,
    end=end,
    initialize=sma_crossover_initialize,
    handle_data=sma_crossover_handle_data,
    capital_base=10_000,
    bundle='raats_daily'
)
equity_sma = result_sma.portfolio_value
returns_sma = result_sma.returns


In [ ]:
# Compute metrics
sharpe_bh = compute_sharpe(returns_bh)
max_dd_bh = compute_max_drawdown(equity_bh)
# For win‑rate we need trade P&L; we'll approximate using transactions
def approximate_trade_pnl(txns):
    if txns.empty:
        return []
    pnl = []
    for sym in txns.index.get_level_values(1).unique():
        sym_txns = txns.xs(sym, level=1)
        pos = 0.0
        cost = 0.0
        for _, row in sym_txns.iterrows():
            amt = row.amount
            price = row.price
            if pos == 0:
                pos = amt
                cost = price * amt
            else:
                pos += amt
                cost += price * amt
                if pos == 0:
                    pnl.append(cost)
                    pos = 0.0
                    cost = 0.0
    return pnl

pnl_bh = approximate_trade_pnl(result_bh.transactions)
win_rate_bh = compute_win_rate(pnl_bh) if pnl_bh else np.nan

pnl_sma = approximate_trade_pnl(result_sma.transactions)
win_rate_sma = compute_win_rate(pnl_sma) if pnl_sma else np.nan

sharpe_sma = compute_sharpe(returns_sma)
max_dd_sma = compute_max_drawdown(equity_sma)

print('=== Buy‑and‑hold ===')
print(f'Sharpe Ratio: {sharpe_bh:.3f}')
print(f'Max Drawdown: {max_dd_bh:.2%}')
print(f'Win Rate: {win_rate_bh:.2%}' if not np.isnan(win_rate_bh) else 'Win Rate: N/A')

print('=== SMA Crossover (50/200) ===')
print(f'Sharpe Ratio: {sharpe_sma:.3f}')
print(f'Max Drawdown: {max_dd_sma:.2%}')
print(f'Win Rate: {win_rate_sma:.2%}' if not np.isnan(win_rate_sma) else 'Win Rate: N/A')


In [ ]:
# Plot equity curves
plt.figure(figsize=(12,6))
plt.plot(equity_bh.index, equity_bh.values, label='Buy‑and‑hold')
plt.plot(equity_sma.index, equity_sma.values, label='SMA 50/200')
plt.title('Equity Curves – Baseline Strategies (AAPL)')
plt.ylabel('Portfolio Value ($)')
plt.xlabel('Date')
plt.legend()
plt.grid(True)
plt.show()
